<a href="https://colab.research.google.com/github/ozhao1323/ECON3916-Statistical-and-Machine-Learning/blob/main/Lab%2012/%5BLab_12_%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/ozhao1323/ECON3916-Statistical-and-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [3]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [4]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:57:46   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [5]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [6]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [8]:
"""
Residual Forensics Dashboard
=============================
Hedonic Pricing OLS Model — Interactive Residual Diagnostics
Tech Economist · Olivia · Northeastern CS/ECON

Dependencies: statsmodels, plotly, pandas, numpy, scikit-learn
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error


# ─────────────────────────────────────────────
# 1. SYNTHETIC HEDONIC DATASET
#    Replace this block with your actual data.
#    The dashboard code downstream is fully
#    model-agnostic — it only needs `results`.
# ─────────────────────────────────────────────
np.random.seed(42)
n = 300

data = pd.DataFrame({
    "sqft":      np.random.normal(1800, 400, n),
    "bedrooms":  np.random.randint(1, 6, n),
    "age":       np.random.randint(0, 50, n),
    "distance":  np.random.exponential(5, n),   # miles to CBD
    "garage":    np.random.binomial(1, 0.6, n),
})

# True hedonic price function + heteroscedastic noise (fan-shaped)
# The variance deliberately scales with sqft to test whether the
# dashboard surfaces the heteroscedasticity pattern visually.
noise = np.random.normal(0, 20_000 + 15 * data["sqft"], n)

data["price"] = (
    80_000
    + 120 * data["sqft"]
    + 10_000 * data["bedrooms"]
    - 800  * data["age"]
    - 5_000 * data["distance"]
    + 12_000 * data["garage"]
    + noise
)


# ─────────────────────────────────────────────
# 2. OLS MODEL (statsmodels)
# ─────────────────────────────────────────────
feature_cols = ["sqft", "bedrooms", "age", "distance", "garage"]
X = sm.add_constant(data[feature_cols])   # add intercept column (β₀)
y = data["price"]

model   = sm.OLS(y, X)
results = model.fit()

print(results.summary())


# ─────────────────────────────────────────────
# 3. RESIDUAL EXTRACTION
#    statsmodels stores everything on the
#    RegressionResultsWrapper object (`results`).
#
#    results.fittedvalues  → ŷ = Xβ̂  (in-sample predictions)
#    results.resid         → ê = y − ŷ  (ordinary residuals)
#
#    Both are pandas Series aligned to the original index,
#    so we can safely concatenate them into a single DataFrame.
# ─────────────────────────────────────────────
fitted    = results.fittedvalues   # ŷ: predicted house prices
residuals = results.resid          # ê = y − ŷ: signed prediction errors

# ─────────────────────────────────────────────
# 4. RMSE
# ─────────────────────────────────────────────
rmse = np.sqrt(mean_squared_error(y, fitted))
print(f"\nRMSE: ${rmse:,.0f}")


# ─────────────────────────────────────────────
# 5. OUTLIER FLAGGING
#    Standardise residuals: z = ê / σ̂_ê
#    Flag any observation where |z| > 2σ.
#    These are candidate structural breaks,
#    data-entry errors, or genuine anomalies.
# ─────────────────────────────────────────────
resid_std  = residuals.std()          # σ̂ of the residual distribution
resid_z    = residuals / resid_std    # z-score of each residual
is_outlier = resid_z.abs() > 2.0      # boolean mask: True → flagged

# Build a single diagnostic DataFrame for plotting
diag_df = pd.DataFrame({
    "fitted":    fitted,
    "residual":  residuals,
    "z_score":   resid_z,
    "outlier":   is_outlier,
    "abs_z":     resid_z.abs(),
    # Hover-friendly label showing actual vs predicted
    "label":     [
        f"Obs {i}<br>ŷ = ${f:,.0f}<br>ê = ${r:,.0f}<br>|z| = {z:.2f}"
        for i, f, r, z in zip(
            residuals.index, fitted, residuals, resid_z.abs()
        )
    ],
})

# Separate inlier and outlier populations for layered rendering
inliers  = diag_df[~diag_df["outlier"]]
outliers = diag_df[ diag_df["outlier"]]

n_outliers = is_outlier.sum()
pct_outliers = 100 * n_outliers / n


# ─────────────────────────────────────────────
# 6. BUILD PLOTLY FIGURE
#    We use go.Figure() with explicit Scatter
#    traces so we can control marker styling,
#    hover templates, and z-ordering precisely.
#    (plotly.express wraps this API but offers
#    less granular control over layering.)
# ─────────────────────────────────────────────
fig = go.Figure()

# ── Layer 1: Inlier scatter ───────────────────
fig.add_trace(go.Scatter(
    x=inliers["fitted"],
    y=inliers["residual"],
    mode="markers",
    name="Within ±2σ",
    text=inliers["label"],
    hovertemplate="%{text}<extra></extra>",
    marker=dict(
        color=inliers["abs_z"],          # encode |z| via colorscale
        colorscale="Blues",
        cmin=0,
        cmax=2,
        size=7,
        opacity=0.72,
        line=dict(width=0.5, color="rgba(255,255,255,0.4)"),
        showscale=True,
        colorbar=dict(
            title=dict(text="|z-score|", side="right"),
            thickness=14,
            len=0.7,
            x=1.02,
        ),
    ),
))

# ── Layer 2: Outlier scatter (crimson) ────────
# Rendered after inliers so they sit on top visually.
fig.add_trace(go.Scatter(
    x=outliers["fitted"],
    y=outliers["residual"],
    mode="markers",
    name=f"Outliers |z| > 2σ  (n={n_outliers})",
    text=outliers["label"],
    hovertemplate="%{text}<extra></extra>",
    marker=dict(
        color="#DC143C",               # Crimson — stark, unambiguous
        size=10,
        symbol="circle",
        opacity=0.92,
        line=dict(width=1.2, color="#8B0000"),   # dark-red border
    ),
))

# ── Layer 3: Zero-reference line (ê = 0) ──────
# A perfectly specified model would have residuals
# randomly scattered symmetrically around this line.
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="rgba(255,255,255,0.55)",
    line_width=1.5,
    annotation_text="ê = 0  (zero-error baseline)",
    annotation_position="bottom right",
    annotation_font=dict(size=11, color="rgba(255,255,255,0.6)"),
)

# ── Layer 4: ±2σ band shading ─────────────────
# Shaded region between -2σ and +2σ for at-a-glance
# reference of the "expected" residual spread.
sigma_band_color = "rgba(100, 160, 230, 0.07)"
fig.add_hrect(
    y0=-2 * resid_std,
    y1= 2 * resid_std,
    fillcolor=sigma_band_color,
    line_width=0,
    annotation_text="±2σ band",
    annotation_position="top left",
    annotation_font=dict(size=10, color="rgba(200,220,255,0.5)"),
)


# ─────────────────────────────────────────────
# 7. LAYOUT & TYPOGRAPHY
# ─────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            "<b>Residual Forensics Dashboard</b>"
            f"<br><sup>Hedonic OLS · RMSE = ${rmse:,.0f} · "
            f"{n_outliers} outliers ({pct_outliers:.1f}%) beyond ±2σ</sup>"
        ),
        x=0.04,
        font=dict(size=20, color="#E8EDF5"),
    ),
    xaxis=dict(
        title="Fitted Values  ŷ  (Predicted Price, $)",
        titlefont=dict(size=13, color="#A8B8D0"),
        tickfont=dict(size=11, color="#A8B8D0"),
        gridcolor="rgba(255,255,255,0.07)",
        zerolinecolor="rgba(255,255,255,0.12)",
        tickprefix="$",
        tickformat=",.0f",
    ),
    yaxis=dict(
        title="Residuals  ê = y − ŷ  (Error, $)",
        titlefont=dict(size=13, color="#A8B8D0"),
        tickfont=dict(size=11, color="#A8B8D0"),
        gridcolor="rgba(255,255,255,0.07)",
        zerolinecolor="rgba(255,255,255,0.12)",
        tickprefix="$",
        tickformat=",.0f",
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=11, color="#C0CDE0"),
        bgcolor="rgba(0,0,0,0)",
    ),
    paper_bgcolor="#0D1117",   # deep navy — matches a dark IDE / notebook
    plot_bgcolor="#111820",
    hovermode="closest",
    margin=dict(l=70, r=90, t=100, b=70),
    height=580,
    width=1000,
)

# ─────────────────────────────────────────────
# 8. EXPORT & DISPLAY
# ─────────────────────────────────────────────
output_html = "residual_forensics_dashboard.html"
fig.write_html(output_html, include_plotlyjs="cdn")
print(f"\nDashboard saved → {output_html}")

fig.show()   # opens browser tab / renders in Jupyter inline


# ─────────────────────────────────────────────────────────────────────────────
# INTERPRETATION GUIDE
# ─────────────────────────────────────────────────────────────────────────────
"""
HOW TO READ THIS PLOT
──────────────────────────────────────────────────────────────────────────────

1. IDEAL PATTERN — Null model of no misspecification
   The residuals should form a horizontal, homogeneous band of uniform
   vertical width centered on ê = 0.  No systematic drift.  Crimson points
   should appear randomly — not clustered in any region.

2. HETEROSCEDASTICITY (fan / funnel shape)
   If the vertical spread of the residual cloud *grows* as ŷ increases,
   you have a fan-out: variance is proportional to fitted price.
   This violates OLS Assumption 5 (homoscedasticity).
   Consequence: OLS standard errors are biased → t-stats and p-values are
   unreliable.  Fix: White/HC3 robust SE, WLS, or log-transform the DV.

3. STRUCTURAL BREAKS (horizontal jump or vertical drift)
   A sudden shift in the residual level across a threshold of ŷ signals
   a regime change — perhaps two sub-markets (luxury vs. starter homes)
   that your linear spec cannot capture.
   Fix: Interaction terms, market-segment dummies, or a piecewise regression.

4. NON-LINEARITY (curved residual envelope)
   A U-shape or inverted-U means a quadratic or log transformation of a
   feature is missing.  The model is systematically under-predicting in
   the middle and over-predicting at the tails (or vice versa).

5. OUTLIER CLUSTERS (crimson concentration)
   A concentration of crimson points in a narrow ŷ-range can indicate:
   - A mis-coded categorical (e.g., garage coded 0/1 reversed)
   - A neighbourhood fixed effect not captured by observables
   - Genuine anomalies (foreclosures, renovations mid-year)
   Always inspect the raw observations behind each cluster.

6. RMSE CALIBRATION
   RMSE = ${rmse:,.0f} means your model's *average* prediction error is
   roughly that magnitude.  Compare to the interquartile range of y to
   assess economic significance, not just statistical fit.
──────────────────────────────────────────────────────────────────────────────
"""

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.571
Model:                            OLS   Adj. R-squared:                  0.563
Method:                 Least Squares   F-statistic:                     78.17
Date:                Mon, 16 Mar 2026   Prob (F-statistic):           6.10e-52
Time:                        20:40:16   Log-Likelihood:                -3660.0
No. Observations:                 300   AIC:                             7332.
Df Residuals:                     294   BIC:                             7354.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       8.999e+04   1.53e+04      5.891      0.0

"\nHOW TO READ THIS PLOT\n──────────────────────────────────────────────────────────────────────────────\n\n1. IDEAL PATTERN — Null model of no misspecification\n   The residuals should form a horizontal, homogeneous band of uniform\n   vertical width centered on ê = 0.  No systematic drift.  Crimson points\n   should appear randomly — not clustered in any region.\n\n2. HETEROSCEDASTICITY (fan / funnel shape)\n   If the vertical spread of the residual cloud *grows* as ŷ increases,\n   you have a fan-out: variance is proportional to fitted price.\n   This violates OLS Assumption 5 (homoscedasticity).\n   Consequence: OLS standard errors are biased → t-stats and p-values are\n   unreliable.  Fix: White/HC3 robust SE, WLS, or log-transform the DV.\n\n3. STRUCTURAL BREAKS (horizontal jump or vertical drift)\n   A sudden shift in the residual level across a threshold of ŷ signals\n   a regime change — perhaps two sub-markets (luxury vs. starter homes)\n   that your linear spec cannot captur